<a href="https://colab.research.google.com/github/PreciousEnahoro/public-education-benchmarking-system/blob/main/notebooks/core_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import packages and GDrive Driver for Colab to access file
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm

Mounted at /content/drive


In [2]:
# Import data

PATH = "/content/drive/MyDrive/lea_year_model_core.csv"
df = pd.read_csv(PATH)
df.head()

,LEAID,year,State,total_enrollment,pct_black,pct_hispanic,pct_white,pct_asian,pct_other_race,poverty_rate_5_17,median_household_income,student_teacher_ratio_clean,ca_rate,urbanicity,grad_rate_2019
0,1300180,2023,GEORGIA,300,0.593333,0.103333,0.236667,0.013333,0.050000,39.8,47850.0,11.583012,0.293,Urban,80.0
1,1300240,2023,GEORGIA,2843,0.017235,0.110447,0.817095,0.011959,0.043264,16.0,70067.0,13.473934,0.302,Urban,87.0
2,1302490,2023,GEORGIA,2589,0.410197,0.164542,0.371572,0.010429,0.039784,25.1,79890.0,10.658707,0.216,Urban,82.0
3,1302640,2023,GEORGIA,703,0.961593,0.008535,0.014225,0.000000,0.015647,42.1,40099.0,8.419162,0.265,Urban,82.0
4,1303660,2023,GEORGIA,735,0.436735,0.048980,0.477551,0.005442,0.031293,28.7,50799.0,12.352941,0.273,Urban,82.0


## **Feature Engineering**

In [3]:
# Keep only what we need
cols = [
    "LEAID","year","ca_rate","poverty_rate_5_17","median_household_income",
    "student_teacher_ratio_clean","pct_black","pct_hispanic","pct_white","pct_asian","pct_other_race",
    "urbanicity","grad_rate_2019", "State"
]
dfm = df[cols].copy()

# Make sure numeric columns are numeric
num_cols = ["ca_rate","poverty_rate_5_17","median_household_income","student_teacher_ratio_clean",
            "pct_black","pct_hispanic","pct_white","pct_asian","pct_other_race","grad_rate_2019"]
for c in num_cols:
    dfm[c] = pd.to_numeric(dfm[c], errors="coerce")

# Urban dummy
dfm["is_urban"] = (dfm["urbanicity"].astype(str).str.lower() == "urban").astype(int)



In [5]:
dfm["year"] = pd.to_numeric(dfm["year"], errors="coerce")
dfm = dfm.dropna(subset=["year"])
dfm["year"] = dfm["year"].astype(int)
#dfm["ca_rate"] = StandardScaler().fit_transform(dfm[["ca_rate"]])

dfm["ca_rate"] = np.where(
    dfm["ca_rate"] > 5,   # anything above 5 is definitely wrong scale
    dfm["ca_rate"] / 100,
    dfm["ca_rate"]
)

df_pool = dfm[dfm["year"].isin([2019, 2023])].copy()

df_std = df_pool.copy()

df_std["year_2023"] = (df_std["year"] == 2023).astype(int)

#Creating interaction terms
df_std["poverty_x_2023"] = df_std["poverty_rate_5_17"] * df_std["year_2023"]
df_std["staff_x_2023"] = df_std["student_teacher_ratio_clean"] * df_std["year_2023"]

df_std.head()

,LEAID,year,ca_rate,poverty_rate_5_17,median_household_income,student_teacher_ratio_clean,pct_black,pct_hispanic,pct_white,pct_asian,pct_other_race,urbanicity,grad_rate_2019,State,is_urban,year_2023,poverty_x_2023,staff_x_2023
0,1300180,2023,0.293,39.8,47850.0,11.583012,0.593333,0.103333,0.236667,0.013333,0.050000,Urban,80.0,GEORGIA,1,1,39.8,11.583012
1,1300240,2023,0.302,16.0,70067.0,13.473934,0.017235,0.110447,0.817095,0.011959,0.043264,Urban,87.0,GEORGIA,1,1,16.0,13.473934
2,1302490,2023,0.216,25.1,79890.0,10.658707,0.410197,0.164542,0.371572,0.010429,0.039784,Urban,82.0,GEORGIA,1,1,25.1,10.658707
3,1302640,2023,0.265,42.1,40099.0,8.419162,0.961593,0.008535,0.014225,0.000000,0.015647,Urban,82.0,GEORGIA,1,1,42.1,8.419162
4,1303660,2023,0.273,28.7,50799.0,12.352941,0.436735,0.048980,0.477551,0.005442,0.031293,Urban,82.0,GEORGIA,1,1,28.7,12.352941


## **Modelling**

###**Core Model**

In [6]:
# (optional) ensure year is int
dfm["year"] = pd.to_numeric(dfm["year"], errors="coerce")
dfm = dfm.dropna(subset=["year"])
dfm["year"] = dfm["year"].astype(int)

df_pool = dfm[dfm["year"].isin([2019, 2023])].copy()

formula_final = """
ca_rate ~ poverty_rate_5_17*C(year)
       + student_teacher_ratio_clean*C(year)
       + pct_black + pct_hispanic
"""

model_final = smf.ols(formula_final, data=df_pool).fit(cov_type="HC3")
print(model_final.summary())

                            OLS Regression Results                            
Dep. Variable:                ca_rate   R-squared:                       0.214
Model:                            OLS   Adj. R-squared:                  0.214
Method:                 Least Squares   F-statistic:                     972.9
Date:                Wed, 22 Apr 2026   Prob (F-statistic):               0.00
Time:                        22:24:24   Log-Likelihood:                 19833.
No. Observations:               24720   AIC:                        -3.965e+04
Df Residuals:                   24712   BIC:                        -3.959e+04
Df Model:                           7                                         
Covariance Type:                  HC3                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

## **Export Data For Tableau and GitHub**



In [7]:
df_pool["pred_ca_rate"] = model_final.predict(df_pool)
df_pool["residual"] = df_pool["ca_rate"] - df_pool["pred_ca_rate"]

# Optional: top-decile flag (for “high-risk districts” list)
df_pool["top_decile_risk"] = (df_pool["pred_ca_rate"] >= df_pool["pred_ca_rate"].quantile(0.90)).astype(int)

tableau_cols = [
    "LEAID", "State", "year","ca_rate","pred_ca_rate","residual",
    "top_decile_risk",
    "poverty_rate_5_17","student_teacher_ratio_clean",
    "pct_black","pct_hispanic"
]

tableau_df = df_pool[tableau_cols].copy()

tableau_df["ca_rate"] = tableau_df["ca_rate"].round(4)
tableau_df["pred_ca_rate"] = tableau_df["pred_ca_rate"].round(4)
tableau_df["residual"] = tableau_df["residual"].round(4)

In [8]:
OUT_DIR = "/content/drive/MyDrive/Absenteeism/core_model"  # change if you want
import os
os.makedirs(OUT_DIR, exist_ok=True)

import pickle

with open(f"{OUT_DIR}/absenteeism_coremodel_final_HC3.pkl", "wb") as f:
    pickle.dump(model_final, f)

coef_df = pd.DataFrame({
    "term": model_final.params.index,
    "coef": model_final.params.values,
    "se_HC3": model_final.bse.values,
    "p_value": model_final.pvalues.values
})
coef_df.to_csv(f"{OUT_DIR}/absenteeism_coremodel_final_coefficients.csv", index=False)

with open(f"{OUT_DIR}/absenteeism_coremodel_final_summary.txt", "w") as f:
    f.write(model_final.summary().as_text())

with open(f"{OUT_DIR}/absenteeism_coremodel_final_formula.txt", "w") as f:
    f.write(formula_final.strip())

tableau_path = f"{OUT_DIR}/absenteeism_tableau_dataset_coremodel.csv"
tableau_df.to_csv(tableau_path, index=False)
tableau_path


'/content/drive/MyDrive/Absenteeism/core_model/absenteeism_tableau_dataset_coremodel.csv'